# RAG Evaluation Pipeline — Local Models (RAGAS 0.4.x)

Uses DenizBank internal LLM (Qwen 3.1) and embedding (Qwen3-Embedding) endpoints.  
No external API keys needed.

## 1 — Install

In [ ]:
!pip install ragas openai httpx pandas --quiet

## 2 — Endpoint Configuration

In [ ]:
# ── LLM Endpoint ──
LLM_BASE_URL = "https://api-qwen-31-infer-tmp-automation-test-3.apps.datascience.prod2.deniz.denizbank.com/v1"
LLM_MODEL = "default"

# ── Embedding Endpoint ──
EMBEDDING_URL = "http://YOUR_EMBEDDING_HOST:PORT/v1/embeddings"  # update this
EMBEDDING_MODEL = "qwen3-embedding"
EMBEDDING_API_KEY = "your-key"  # or "not-needed" if no auth

## 3 — LLM Setup

Your vLLM endpoint is OpenAI-compatible, so `llm_factory` works directly.  
The `chat_template_kwargs` (like `enable_thinking`) are vLLM-specific extras — 
RAGAS won't pass those, but for eval-judge calls it doesn't matter.

In [ ]:
from openai import AsyncOpenAI
from ragas.llms import llm_factory

llm_client = AsyncOpenAI(
    base_url=LLM_BASE_URL,
    api_key="not-needed",
)

evaluator_llm = llm_factory(
    LLM_MODEL,
    provider="openai",
    client=llm_client,
)

print(f"\u2705 LLM ready: {LLM_MODEL} @ {LLM_BASE_URL}")

## 4 — Embedding Setup

Your embedding endpoint accepts OpenAI-compatible batch requests, 
so we can use `embedding_factory` with a custom `AsyncOpenAI` client.

Also providing a raw `BaseRagasEmbedding` subclass that matches your 
`get_embeddings_batch` exactly — use whichever works cleaner with your infra.

### Option A: Via `embedding_factory` (recommended)

In [ ]:
from ragas.embeddings.base import embedding_factory

# Extract base URL (everything before /embeddings)
# e.g., if EMBEDDING_URL = "http://host:port/v1/embeddings"
#        then base = "http://host:port/v1"
EMB_BASE = EMBEDDING_URL.rsplit("/embeddings", 1)[0]

emb_client = AsyncOpenAI(
    base_url=EMB_BASE,
    api_key=EMBEDDING_API_KEY,
)

evaluator_embeddings = embedding_factory(
    provider="openai",
    model=EMBEDDING_MODEL,
    client=emb_client,
)

print(f"\u2705 Embeddings ready: {EMBEDDING_MODEL} @ {EMB_BASE}")

### Option B: Direct wrapper matching your `get_embeddings_batch`

In [ ]:
# import httpx
# from ragas.embeddings.base import BaseRagasEmbedding
#
#
# class DenizBankEmbedding(BaseRagasEmbedding):
#     """Wraps the internal Qwen3-Embedding endpoint."""
#
#     def __init__(self, url: str, model: str, api_key: str = "not-needed"):
#         super().__init__()
#         self.url = url
#         self.model = model
#         self.api_key = api_key
#         self._client = httpx.Client(timeout=120.0)
#         self._async_client = httpx.AsyncClient(timeout=120.0)
#
#     def _headers(self):
#         return {
#             "Authorization": f"Bearer {self.api_key}",
#             "Content-Type": "application/json",
#         }
#
#     def embed_text(self, text: str, **kwargs) -> list[float]:
#         resp = self._client.post(
#             self.url,
#             json={"model": self.model, "input": [text]},
#             headers=self._headers(),
#         )
#         resp.raise_for_status()
#         return resp.json()["data"][0]["embedding"]
#
#     async def aembed_text(self, text: str, **kwargs) -> list[float]:
#         resp = await self._async_client.post(
#             self.url,
#             json={"model": self.model, "input": [text]},
#             headers=self._headers(),
#         )
#         resp.raise_for_status()
#         return resp.json()["data"][0]["embedding"]
#
#     def embed_texts(self, texts: list[str], **kwargs) -> list[list[float]]:
#         """Batch embed — matches your get_embeddings_batch logic."""
#         resp = self._client.post(
#             self.url,
#             json={"model": self.model, "input": texts},
#             headers=self._headers(),
#         )
#         resp.raise_for_status()
#         data = resp.json()["data"]
#         return [item["embedding"] for item in sorted(data, key=lambda x: x["index"])]
#
#     async def aembed_texts(self, texts: list[str], **kwargs) -> list[list[float]]:
#         resp = await self._async_client.post(
#             self.url,
#             json={"model": self.model, "input": texts},
#             headers=self._headers(),
#         )
#         resp.raise_for_status()
#         data = resp.json()["data"]
#         return [item["embedding"] for item in sorted(data, key=lambda x: x["index"])]
#
#
# evaluator_embeddings = DenizBankEmbedding(
#     url=EMBEDDING_URL,
#     model=EMBEDDING_MODEL,
#     api_key=EMBEDDING_API_KEY,
# )

## 5 — Smoke Test (Verify Connectivity)

In [ ]:
import httpx, requests, json

# Test LLM
print("Testing LLM endpoint...")
resp = requests.post(
    f"{LLM_BASE_URL}/chat/completions",
    json={
        "model": LLM_MODEL,
        "messages": [{"role": "user", "content": "Merhaba, test."}],
        "chat_template_kwargs": {"enable_thinking": False},
        "temperature": 0.1,
    },
)
print(f"  LLM status: {resp.status_code}")
if resp.ok:
    print(f"  Response: {json.loads(resp.text)['choices'][0]['message']['content'][:100]}...")

# Test Embedding
print("\nTesting embedding endpoint...")
resp2 = httpx.post(
    EMBEDDING_URL,
    json={"model": EMBEDDING_MODEL, "input": ["test"]},
    headers={"Authorization": f"Bearer {EMBEDDING_API_KEY}"},
    timeout=30.0,
)
print(f"  Embedding status: {resp2.status_code}")
if resp2.is_success:
    dim = len(resp2.json()["data"][0]["embedding"])
    print(f"  Embedding dimension: {dim}")

print("\n\u2705 Both endpoints reachable")

## 6 — Build Evaluation Dataset

Replace these samples with your actual RAG pipeline outputs.

In [ ]:
from ragas import SingleTurnSample, EvaluationDataset

samples = [
    SingleTurnSample(
        user_input="Basel III kapsam\u0131nda sermaye yeterlilik oranlar\u0131 nelerdir?",
        response=(
            "Basel III kapsam\u0131nda bankalar minimum %4.5 CET1, "
            "%6 Tier 1 ve %8 toplam sermaye oran\u0131 tutmak zorundad\u0131r."
        ),
        retrieved_contexts=[
            "Basel III'e g\u00f6re asgari \u00c7ekirdek Sermaye (CET1) oran\u0131 %4.5'tir. "
            "Asgari Tier 1 sermaye oran\u0131 %6, toplam sermaye oran\u0131 ise %8'dir.",
            "Basel III ayr\u0131ca %2.5'lik sermaye koruma tamponu getirmi\u015ftir.",
        ],
        reference=(
            "Basel III minimum CET1 %4.5, Tier 1 %6, toplam sermaye %8 "
            "ve %2.5 sermaye koruma tamponu \u00f6ng\u00f6r\u00fcr."
        ),
    ),
    SingleTurnSample(
        user_input="FAISS yakla\u015f\u0131k en yak\u0131n kom\u015fu aramas\u0131n\u0131 nas\u0131l yapar?",
        response=(
            "FAISS, b\u00fcy\u00fck \u00f6l\u00e7ekli vekt\u00f6r veri setlerinde h\u0131zl\u0131 ANN aramas\u0131 i\u00e7in "
            "ters dosya indeksleri (IVF) ve \u00e7arp\u0131m nicelemeyi (PQ) birle\u015ftirir."
        ),
        retrieved_contexts=[
            "FAISS, verimli benzerlik aramas\u0131 i\u00e7in IVF ve PQ uygular.",
            "Milyar \u00f6l\u00e7ekli veri setlerinde FAISS, alt-do\u011frusal arama s\u00fcresi sa\u011flar.",
        ],
        reference="FAISS, \u00f6l\u00e7ekte ANN aramas\u0131 i\u00e7in IVF ve \u00e7arp\u0131m niceleme kullan\u0131r.",
    ),
    SingleTurnSample(
        user_input="LightGBM ve XGBoost aras\u0131ndaki fark nedir?",
        response=(
            "LightGBM histogram tabanl\u0131 b\u00f6lme ve yaprak bazl\u0131 b\u00fcy\u00fcme kullan\u0131r, daha h\u0131zl\u0131d\u0131r. "
            "XGBoost varsay\u0131lan olarak seviye bazl\u0131 b\u00fcy\u00fcme ve kesin a\u00e7g\u00f6zl\u00fc b\u00f6lme kullan\u0131r."
        ),
        retrieved_contexts=[
            "LightGBM histogram tabanl\u0131 karar a\u011fac\u0131 \u00f6\u011frenimi kullan\u0131r ve yaprak bazl\u0131 b\u00fcy\u00fcr.",
            "XGBoost varsay\u0131lan olarak kesin a\u00e7g\u00f6zl\u00fc algoritma kullan\u0131r ve seviye bazl\u0131 b\u00fcy\u00fcr. "
            "tree_method='hist' ile histogram tabanl\u0131 y\u00f6ntemleri de destekler.",
        ],
        reference=(
            "LightGBM histogram tabanl\u0131 yaprak bazl\u0131 b\u00fcy\u00fcme ile daha h\u0131zl\u0131d\u0131r; "
            "XGBoost varsay\u0131lan olarak seviye bazl\u0131 kesin b\u00f6lme yapar."
        ),
    ),
]

eval_dataset = EvaluationDataset(samples=samples)
print(f"\u2705 Dataset: {len(samples)} samples")

### Load from CSV (alternative)

In [ ]:
# import json
# import pandas as pd
#
# df = pd.read_csv("rag_outputs.csv")
# df["retrieved_contexts"] = df["retrieved_contexts"].apply(json.loads)
#
# eval_dataset = EvaluationDataset(
#     samples=[SingleTurnSample(**row.to_dict()) for _, row in df.iterrows()]
# )

## 7 — Core Metrics

| Metric | What it measures | Needs `reference`? |
|--------|------------------|--------------------|
| Faithfulness | Response grounded in retrieved contexts | No |
| ResponseRelevancy | Response relevant to the question (uses embeddings) | No |
| ContextPrecision | Retrieved chunks actually relevant | No |
| FactualCorrectness | Response matches ground truth | **Yes** |

In [ ]:
from ragas.metrics import (
    Faithfulness,
    ResponseRelevancy,
    LLMContextPrecisionWithoutReference,
    FactualCorrectness,
)

core_metrics = [
    Faithfulness(),
    ResponseRelevancy(),
    LLMContextPrecisionWithoutReference(),
    FactualCorrectness(),
]

## 8 — Run Evaluation

> **Critical:** Always pass `embeddings=evaluator_embeddings`.  
> Without it, RAGAS falls back to OpenAI and throws auth errors.

In [ ]:
from ragas import evaluate

results = evaluate(
    dataset=eval_dataset,
    metrics=core_metrics,
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
)

print(results)

In [ ]:
import pandas as pd

results_df = results.to_pandas()

metric_cols = [
    "faithfulness",
    "response_relevancy",
    "llm_context_precision_without_reference",
    "factual_correctness",
]

results_df[["user_input"] + metric_cols]

## 9 — Custom Metrics

### 9a — AspectCritic (Binary Pass/Fail)

In [ ]:
from ragas.metrics import AspectCritic

hallucination_check = AspectCritic(
    name="no_hallucination",
    definition=(
        "Yan\u0131t YALNIZCA al\u0131nan ba\u011flamlarda desteklenen bilgileri i\u00e7erir. "
        "Uydurma say\u0131lar, tarihler veya iddialar yoktur."
    ),
    llm=evaluator_llm,
)

completeness_check = AspectCritic(
    name="completeness",
    definition=(
        "Yan\u0131t, kullan\u0131c\u0131n\u0131n sorusunun t\u00fcm k\u0131s\u0131mlar\u0131n\u0131 tam olarak ele al\u0131r "
        "ve ba\u011flamlardaki \u00f6nemli bilgileri atlamaz."
    ),
    llm=evaluator_llm,
)

custom_results = evaluate(
    dataset=eval_dataset,
    metrics=[hallucination_check, completeness_check],
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
)

custom_results.to_pandas()[["user_input", "no_hallucination", "completeness"]]

### 9b — DiscreteMetric (Numeric 1-5 Scale)

In [ ]:
from ragas.metrics import DiscreteMetric

technical_depth = DiscreteMetric(
    name="technical_depth",
    allowed_values=list(range(1, 6)),
    prompt="""Yan\u0131t\u0131n teknik derinli\u011fini puanla.
1 = Y\u00fczeysel, detay yok
2 = Temel, n\u00fcanslar\u0131 eksik
3 = Yeterli
4 = \u0130yi derinlik, do\u011fru teknik detaylar
5 = Uzman d\u00fczeyinde hassasiyet

Kullan\u0131c\u0131 sorusu: {user_input}
Yan\u0131t: {response}

Sadece say\u0131 ile cevap ver (1-5).""",
)

score = await technical_depth.ascore(
    llm=evaluator_llm,
    user_input=samples[0].user_input,
    response=samples[0].response,
)
print(f"Technical depth: {score.value}/5 \u2014 {score.reason}")

## 10 — Batch Evaluation Helper

Wire this into your actual RAG pipeline. Replace `run_rag_pipeline` with 
your retriever + `generate_response` call.

In [ ]:
import requests, json
from typing import Optional


def run_rag_pipeline(query: str, retriever) -> dict:
    """
    Replace with your actual pipeline.
    1. Retrieve contexts via your retriever (FAISS, etc.)
    2. Call generate_response with query + context
    """
    # contexts = retriever.search(query, top_k=5)
    # context_str = "\n".join(contexts)
    #
    # resp = requests.post(
    #     f"{LLM_BASE_URL}/chat/completions",
    #     json={
    #         "model": LLM_MODEL,
    #         "messages": [
    #             {"role": "system", "content": "Sadece verilen metinlerdeki bilgileri kullanarak cevap ver."},
    #             {"role": "user", "content": f"Metin: {context_str}. Soru: {query}. Cevap:"},
    #         ],
    #         "chat_template_kwargs": {"enable_thinking": False},
    #         "temperature": 0.1,
    #     },
    # )
    # response_text = json.loads(resp.text)["choices"][0]["message"]["content"]
    #
    # return {"response": response_text, "retrieved_contexts": contexts}
    raise NotImplementedError("Wire up your retriever + LLM here")


def evaluate_pipeline(
    queries: list[str],
    retriever,
    references: Optional[list[str]] = None,
    metrics=None,
) -> pd.DataFrame:
    """End-to-end: run queries -> build dataset -> evaluate -> DataFrame."""
    _samples = []
    for i, q in enumerate(queries):
        out = run_rag_pipeline(q, retriever)
        _samples.append(SingleTurnSample(
            user_input=q,
            response=out["response"],
            retrieved_contexts=out["retrieved_contexts"],
            reference=references[i] if references else None,
        ))

    res = evaluate(
        dataset=EvaluationDataset(samples=_samples),
        metrics=metrics or core_metrics,
        llm=evaluator_llm,
        embeddings=evaluator_embeddings,
    )
    return res.to_pandas()

## 11 — Results Analysis & Export

In [ ]:
def analyze_results(df: pd.DataFrame, cols: list[str], threshold: float = 0.7):
    """Aggregate stats and flag weak samples."""
    summary = df[cols].describe().T[["mean", "std", "min", "25%", "50%"]]
    summary.columns = ["mean", "std", "min", "p25", "median"]

    for col in cols:
        df[f"{col}_flag"] = df[col] < threshold

    flagged = df[df[[f"{c}_flag" for c in cols]].any(axis=1)]
    print(f"\n\u26a0\ufe0f  {len(flagged)}/{len(df)} samples below {threshold}")
    if len(flagged) > 0:
        display(flagged[["user_input"] + cols])

    return summary


summary = analyze_results(results_df, metric_cols)
print("\n\U0001f4ca Metric Summary:")
summary

In [ ]:
results_df.to_csv("eval_results.csv", index=False)
summary.to_csv("eval_summary.csv")
print("\u2705 Exported to eval_results.csv / eval_summary.csv")

## Notes & Caveats

- **`chat_template_kwargs`**: RAGAS makes its own LLM calls internally for judging — it won't pass `enable_thinking: False`. This is fine; RAGAS prompts are structured enough that thinking mode shouldn't matter for scoring. If you see issues, add `extra_body={"chat_template_kwargs": {"enable_thinking": False}}` when constructing the `AsyncOpenAI` client.
- **Turkish prompts in custom metrics**: The `AspectCritic` and `DiscreteMetric` definitions are in Turkish so the local Qwen model gives more consistent scoring for Turkish RAG outputs. Switch to English if evaluating English content.
- **`ResponseRelevancy`** uses embeddings internally — if `qwen3-embedding` quality is low on your domain, this metric will be noisy. The LLM-only metrics (Faithfulness, FactualCorrectness) are more reliable.
- **Always** pass `embeddings=evaluator_embeddings` to `evaluate()`. RAGAS silently falls back to OpenAI otherwise.
- For larger eval sets, use `RunConfig(max_workers=N)` to manage concurrency against your internal endpoints.